# GreenGate: a live demonstration

**A confidence-aware cascading framework for optimizing energy and cost in large language and multimodal model inference**

P. T. R. Hemachandra · 28998 · BSc (Hons) Computer Science · NSBM Green University

`pip install greengate` · [github.com/Rasath16/GreenGate](https://github.com/Rasath16/GreenGate) · MIT licence

---

### The one idea

A cascade asks a cheap model first and escalates to an expensive one only when the cheap model is unsure.
**The cheap model runs on every single query.** So each query that escalates is paid for *twice* , once for the
answer that was thrown away, and once for the answer that was kept.

No published cascading system counts that second payment. This library does, and counting it changes the answer.

---

### How to run this notebook

| | |
|---|---|
| **Section 0** | Setup. Run this **before** the viva , it downloads model weights. |
| **Sections 1–8** | The demonstration. Every cell runs in seconds once setup is done. |

**Settings → Accelerator → GPU T4 x2** before running anything. Without a GPU the library falls back to an
estimate and says so, which is honest but not what we want to show.

---
## 0 · Setup

*Run these four cells on arrival, not during the presentation.*

In [ ]:
# 0.1  Install the library from the Python Package Index.
#      This is the real published package, not a local copy.
#      The [gpu] extra adds pynvml, for reading the board power sensor, and
#      bitsandbytes, so the 7B large tier can be loaded in 4-bit.
!pip install -q "greengate[gpu]"

# If this notebook has been run before in the same session, restart the kernel
# after installing: transformers caches which backends are available at import.

import greengate
print('greengate version :', greengate.__version__)
print('public API        :', greengate.__all__)

In [ ]:
# 0.2  Confirm we have a real GPU. The profiler reads the board's own power
#      sensor through NVML; with no GPU it falls back to an estimate and labels it.
import torch, subprocess
print('CUDA available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device         :', torch.cuda.get_device_name(0))
    print(subprocess.run(['nvidia-smi','--query-gpu=name,power.draw,power.limit,memory.total',
                          '--format=csv'], capture_output=True, text=True).stdout)

In [ ]:
# 0.3  Choose the two tiers. These are the models from the thesis evaluation:
#      a small open-weight model that exposes token probabilities, and a 7B
#      large tier quantised to 4-bit so both fit on one T4.
SMALL = 'Qwen/Qwen2.5-0.5B-Instruct'
LARGE = 'mistralai/Mistral-7B-Instruct-v0.2'

# Both are in the library's shipped calibration registry, so no fitting is needed.
import json, importlib.resources as res
presets = json.loads(res.files('greengate').joinpath('presets.json').read_text())
for m in (SMALL, LARGE):
    p = presets.get(m)
    print(f'{m:45s} temperature={p["temperature"]:<8} ECE {p["ece_before"]} -> {p["ece_after"]}')

In [ ]:
# 0.4  Load both tiers. This downloads weights the first time (a few minutes).
#      Do this BEFORE the viva so the demonstration itself is instant.
import time
t0 = time.time()
gw = greengate.GreenGate(small=SMALL, large=LARGE,
                         large_is_api=False,   # both tiers local, both measured
                         large_4bit=True,      # 7B in 4-bit fits beside the small tier
                         max_new_tokens=60)
print(f'both tiers loaded in {time.time()-t0:.0f} s')

# A single warm-up call so the first timed measurement is not polluted by
# CUDA kernel compilation.
_ = gw.route('Say hello.')
gw._session = type(gw._session)()   # reset the counters after warm-up
print('ready')

---
## 1 · The instrument

Before any routing, the library is a **measuring device**. `measure()` wraps *any* code , including
inference this library does not run , and reports what it cost.

> **Think of it like this.** A fuel gauge you can fit to a car somebody else built.

In [ ]:
with greengate.measure(n_queries=1, label='someone else\'s pipeline') as m:
    out = gw._small.generate('Explain what a cache miss is, in two sentences.')

print(m.report())
print()
print('energy, joules      :', round(m.energy_joules, 3))
print('carbon, grams CO2   :', round(m.carbon_grams, 8))
print('average power, watts:', round(m.avg_power_w, 1))
print('measurement mode    :', m.mode)

`mode` is the honesty field. On this GPU it reads **`gpu (pynvml)`** , the board's own power sensor,
polled every 100 ms and integrated over the run. On a laptop with no GPU it reads `cpu (estimated)`
and the library does not pretend otherwise.

Repeatability was measured at **1.4%** on text and **0.5%** on vision by re-running identical workloads.

---
## 2 · The instrument refuses to mis-measure

Device power cannot be split between two processes sharing a GPU. During the vision experiments one run
was launched twice by mistake; whole-device power attributed each run's consumption partly to the other
and **one tier came out 42% too high**. It was caught because that tier is measured in both runs and the
values disagreed.

Every measurement now records whether it had the accelerator to itself.

In [ ]:
with greengate.measure(n_queries=1, label='exclusivity check') as m:
    _ = gw._small.generate('What is 17 times 23?')

print('had the GPU to itself :', m.exclusive)
print('other compute processes:', m.shared_with or 'none')
print()
print('With strict=True the measurement raises ContendedMeasurement instead of')
print('quietly returning a number that cannot be attributed to this process.')

---
## 3 · The accounting rule

A cascade runs the small tier on **every** query and the large tier on the escalated fraction. So against
simply using the large model for everything:

$$\text{saving} \;=\; 1 \;-\; \frac{C_{\text{small}}}{C_{\text{large}}} \;-\; e$$

where $e$ is the escalation rate. The cascade only saves while $e$ stays below $1 - C_s/C_l$.

> **Think of it like this.** If the small model is 47% of the price, that leaves 53% of headroom.
> Every query you escalate eats one point of it.

`should_cascade()` applies the rule to measured costs and returns a verdict in plain words.

In [ ]:
from greengate import should_cascade

# Small model costs 47, large costs 100. Three different kinds of traffic.
for label, rate in [('mostly easy questions', 0.10),
                    ('mixed traffic',          0.30),
                    ('half are hard',          0.53),
                    ('ALL questions are hard', 1.00)]:
    v = should_cascade(47, 100, rate)
    print(f'{label:24s} escalate {rate:4.0%}  ->  {v["verdict"]}')

Read the last line. **If every query is hard, cascading is the worst thing you can do** , you pay for the
small model on every query and throw every one of those answers away.

That is the finding no previous cascading paper can express, because none of them charges for the
discarded run.

---
## 4 · Decide before deploying anything

A developer should not have to deploy a cascade to find out whether it helps. `audit()` runs **both**
tiers over a sample of their own queries, so the cost ratio is *measured* rather than assumed, and reports
the threshold that achieves each escalation rate together with the saving it would produce.

This also solves the cold-start problem: the threshold is learned from real traffic in the same pass.

In [ ]:
SAMPLE = [
    'What is the capital of France?',
    'Convert 45 degrees Celsius to Fahrenheit.',
    'What does HTTP status code 404 mean?',
    'Name three primary colours.',
    'Write a SQL query to find the second highest salary in a table.',
    'Explain the difference between a process and a thread.',
    'Explain why transformer attention is quadratic in sequence length, and what has been proposed to reduce it.',
    'Design a caching strategy for a news site whose traffic spikes unpredictably, and justify the eviction policy.',
]

report = gw.audit(SAMPLE)

The `recommendation` line is the product. It does **not** promise a saving. It tells the developer
whether they are getting one, including when the honest answer is no.

Note that `audit()` reports **cost, not quality**. The library contains no judge, and says so rather than
inventing a quality number.

---
## 4b · What the signal actually looks like on this traffic

Before routing on it, look at it. The library ships a *calibrated* signal, because across 500 judged
queries calibration raised discrimination from 0.533 to 0.595. But that calibration temperature was
fitted on four-way multiple-choice logits, and here it is being applied to a 152,000-token vocabulary.

Watch what that does.

In [ ]:
import torch

PROBE = [
    'What is the capital of France?',
    'Name three primary colours.',
    'What does HTTP status code 404 mean?',
    'Explain why transformer attention is quadratic in sequence length.',
    'Design a caching strategy for a news site with unpredictable spikes.',
]

tok, mod = gw._small.tokenizer, gw._small.model

def mean_entropy(q, T, n=25):
    text = tok.apply_chat_template([{'role':'user','content':q}],
                                   add_generation_prompt=True, tokenize=False)
    ids = tok(text, return_tensors='pt')['input_ids'].to(mod.device)
    es = []
    with torch.no_grad():
        for _ in range(n):
            lg = mod(ids).logits[0, -1].float()
            pr = torch.softmax(lg / T, -1)
            es.append(float(-(pr * torch.log2(pr.clamp_min(1e-12))).sum()))
            nxt = lg.argmax().view(1, 1)
            ids = torch.cat([ids, nxt], 1)
            if nxt.item() == tok.eos_token_id:
                break
    return sum(es) / len(es)

T_CAL = gw.temperature
raw, cal = [], []
print(f"{'question':52s}{'raw':>9s}{'calibrated':>13s}")
for q in PROBE:
    r, c = mean_entropy(q, 1.0), mean_entropy(q, T_CAL)
    raw.append(r); cal.append(c)
    print(f'{q[:50]:52s}{r:9.3f}{c:13.3f}')

print()
print(f"{'spread, highest minus lowest':52s}{max(raw)-min(raw):9.3f}{max(cal)-min(cal):13.3f}")
print(f"{'spread as a share of the mean':52s}"
      f'{(max(raw)-min(raw))/(sum(raw)/len(raw)):8.0%}{(max(cal)-min(cal))/(sum(cal)/len(cal)):13.0%}')

**Read the raw column.** It orders the questions exactly as intuition would: the capital of France is the
most confident, the open-ended conceptual question the least. **Now read the calibrated column.** Everything
is compressed into a few percent of its own mean, and the ordering is scrambled.

That is not a bug, it is a scale mismatch I can name precisely: the temperature was fitted on a four-way
choice distribution and is being applied to a 152,000-token one, which pushes it toward uniform.

It also *is* my second finding, visible live. On open-ended generation this signal is weak: I measure an
AUROC of 0.595 across 500 judged queries, barely above chance, and my thesis bounds any routing gain at
0.65 points of quality. A weak signal sampled eight times looks like a coin flip, which is exactly why the
thesis reports confidence intervals rather than single numbers.

**What never depends on the signal is the accounting.** That is the next section, and it is the contribution.

---
## 5 · Routing, live

Now set the gate at the threshold the audit found for a 30% escalation budget, and send real questions
through. Watch the `wasted` figure on the escalated one.

In [ ]:
row = [r for r in report['rows'] if r['target_rate'] == 0.3][0]
gw.config(threshold=row['threshold'])
print(f"gate set at {row['threshold']:.4f}  (the audit's 30% operating point)\n")

QUESTIONS = [
    'What is the capital of France?',
    'Design a caching strategy for a news site whose traffic spikes unpredictably, and justify the eviction policy.',
]

for q in QUESTIONS:
    r = gw.route(q)
    print('=' * 92)
    print(q)
    print('-' * 92)
    print(f'  signal     {r.signal:.4f}      threshold {r.threshold:.4f}')
    print(f'  decision   {r.decision}')
    print(f'  answered by{"":2s}{r.large_model if r.decision.startswith("ESCALATE") else r.small_model}')
    print(f'  carbon     {r.carbon_g:.8f} g total')
    print(f'  discarded  {r.wasted_carbon_g:.8f} g   <-- the run that was thrown away')
    print(f'  latency    {r.latency_s:.2f} s')
    print()
    print('  ' + r.response.strip()[:300].replace('\n', '\n  '))
    print()

On the escalated query, `discarded` is non-zero. **That number is the contribution of this thesis.**
Every other cascading tool reports the query as costing only the large-model run.

---
## 6 · The session verdict

`profile()` reports what this session actually emitted, applies the break-even condition to the per-tier
costs it measured, and returns a verdict in plain words.

In [ ]:
p = gw.profile()
for k, v in p.items():
    print(f'  {k:32s} {v}')

If the large tier has not run often enough for its mean cost to mean anything, `verdict` says **what is
still missing** rather than guessing , for example *"no verdict yet: needs 5 more escalations to measure
the large tier"*. An instrument that does not know should say so.

---
## 7 · The carbon ceiling

A developer can cap emissions over a rolling window. When escalating would breach the cap, the router
returns the small-tier answer and records the degradation, rather than silently overrunning the budget
or failing the request.

In [ ]:
gw.config(budget_g=0.0005)   # a deliberately tiny ceiling, to force the behaviour

for q in ['Explain why transformer attention is quadratic in sequence length.',
          'Design a fault-tolerant message queue and justify the delivery guarantees.']:
    r = gw.route(q)
    print(f'{r.decision:16s}  signal {r.signal:.3f}   carbon {r.carbon_g:.8f} g')

print()
print('queries blocked by the carbon ceiling :', gw.profile()['budget_blocked'])
print()
print('LOCAL(budget) means the gate wanted to escalate and the ceiling refused.')
print('The answer still comes back. Quality degrades gracefully instead of the')
print('service failing or the budget being silently exceeded.')

---
## 8 · Two things added after the evaluation

### Services that batch their requests

Under continuous batching a device-level meter cannot attribute energy to one request , but it measures a
*window of work* correctly. `ServiceLedger` records each window with the tier that served it, then derives
the per-query costs, the realised escalation rate and the same break-even verdict from the totals.

This is the only honest option when per-request attribution is impossible.

In [ ]:
from greengate import ServiceLedger

ledger = ServiceLedger()
# Each call records one measured window of batched work and the tier that served it.
ledger.add(tier='small', queries=200, carbon_grams=0.0120)
ledger.add(tier='small', queries=190, carbon_grams=0.0115)
ledger.add(tier='large', queries=60,  carbon_grams=0.0620)
ledger.add(tier='large', queries=58,  carbon_grams=0.0605)

for k, v in ledger.report().items():
    print(f'  {k:32s} {v}')

### Where in the world the machine is

Grid carbon intensity varies by more than **twenty times** between regions. The default is the world
average of 475 gCO2/kWh, which could misstate a developer's real figure by an order of magnitude.

In [ ]:
for region, gco2 in [('Norway', 30), ('France', 56), ('world average', 475), ('India', 713)]:
    j = 1000.0   # the same one kilojoule of inference, everywhere
    kwh = j / 3_600_000
    grams = kwh * 1.2 * gco2
    print(f'  {region:16s} {gco2:>4} gCO2/kWh  ->  {grams:.6f} g for the same work')

print()
print('Set it with:  greengate.GreenGate(small=..., large=..., carbon_intensity=56)')

---
## What this demonstration showed

| | |
|---|---|
| **1** | The library measures inference it does not run, from the GPU's own power sensor |
| **2** | It refuses to report a figure it cannot attribute, after that failure mode was found in my own data |
| **3** | The break-even condition decides, in plain words, whether cascading can pay at all |
| **4** | `audit()` answers that question on a developer's own traffic *before* anything is deployed |
| **5** | Routing charges escalated queries for both runs, and names the discarded one |
| **6** | `profile()` returns the session verdict, or says what it still needs to know |
| **7** | A carbon ceiling degrades quality gracefully instead of overrunning |
| **8** | Batched services and regional grids are handled honestly rather than ignored |

---

**It does not promise a saving. It tells you whether you are getting one, including when the honest
answer is no.** That is the difference between a claim and an instrument.

---

*Every figure in the thesis can be recomputed from the public repository with no GPU and no API key:*

```
git clone https://github.com/Rasath16/GreenGate
cd GreenGate && python bootstrap_results.py && python eval_crossfamily.py
```